# 90—Publish snapshot to Cloud Storage

### ⚠️ ROI maintainers only. This is not a student notebook.

It writes to an ROI-owned bucket and will throw a permissions error for anyone else. It is in
the repo because it belongs next to the thing it mirrors, not because students should run it.

## The important design decision

**This notebook does not reimplement the transformations.** It fetches
`01_load_explore.ipynb` by raw URL, strips the cells that write to BigQuery, executes the rest
once per metro with `METRO` overridden, and exports the resulting dataframes to Parquet.

That guarantees the snapshot cannot drift from what students actually get. The moment someone
edits the student notebook, this picks the change up on the next run. A hand-maintained copy
of the same logic would be wrong within a week.

## Why the snapshot exists at all

`01_load_explore.ipynb` pulls live from irs.gov, foodsafety.gov, and Seattle's Socrata API.
Three external dependencies, on a day when 150 people hit them inside the same ten minutes.
If any one is down or rate-limiting, the notebook fails for the entire room at once.

`scripts/load.sh` rebuilds every table from this snapshot instead, so a single upstream outage
costs a team five minutes rather than their afternoon.

## Bucket layout

```
gs://class-demo/a4i-2026/challenge-2-food-equity/<metro>/<table>/*.parquet
```

`a4i-2026` is the generic top level every challenge shares.
The bucket needs `allUsers:objectViewer` so students can read anonymously from their own
projects—`load.sh` does not authenticate against it.

In [ ]:
# --- Configuration ---------------------------------------------------------
BUCKET  = "class-demo"
PREFIX  = "a4i-2026/challenge-2-food-equity"

NOTEBOOK_URL = ("https://raw.githubusercontent.com/haggman/"
                "A4I2026-challenge-2-food-equity/main/notebooks/01_load_explore.ipynb")

# Every metro we publish, with the two facts that let us prove afterwards that
# the metro override actually took: the state its organizations must come from,
# and the two-digit state FIPS every one of its census tract ids must start with.
#
# This pairing exists because of a real incident. A refactor of the student
# config cell broke the METRO substitution below, the substitution failed
# silently, and all nine metros published Chicago's data. Everything loaded,
# every row count looked plausible, and the snapshot was wrong. Row counts do
# not tell you whose city you are looking at. These two columns do.
#
# Adding a metro here is the only change needed - load.sh discovers what exists
# by listing the bucket.
METROS = {
    "Chicago":      ("IL", "17"),
    "Dallas":       ("TX", "48"),
    "Seattle":      ("WA", "53"),
    "Philadelphia": ("PA", "42"),
    "Atlanta":      ("GA", "13"),
    "Houston":      ("TX", "48"),
    "Denver":       ("CO", "08"),
    "New York":     ("NY", "36"),
    "Phoenix":      ("AZ", "04"),
}

TABLES = {
    "recipients":         "out",              # variable name in the student notebook
    "surplus_postings":   "surplus",
    "shelf_life":         "shelf_life",
    "tract_demographics": "tracts",
}

import google.auth
credentials, PROJECT_ID = google.auth.default()
print(f"Publishing from project: {PROJECT_ID}")
print(f"Metros: {', '.join(METROS)}")
print(f"Target: gs://{BUCKET}/{PREFIX}/<metro>/<table>/")

## Fetch the student notebook and extract its code

We take the code cells and drop three kinds we must not execute:

- **`%%bigquery` magics** — they need a live dataset, and we are not loading anything here.
- **The `load_table` calls** — the whole point is to export dataframes rather than write
  BigQuery tables. The surrounding cell still runs, because that is where `out` is built.
- **Appendix B** — the standalone city-viability tester. It is a diagnostic, not part of the
  pipeline, and running it nine times would re-fetch irs.gov nine more times for nothing. It
  carries a marker comment naming this notebook, so the skip survives someone retitling it.

We then locate the config cell **once**, up front, and assert we found exactly one. The metro
override rewrites a line in that cell, and a rewrite that silently matches nothing is the
failure mode this whole notebook is most exposed to — so it is checked rather than assumed.

In [ ]:
import re
import requests
import nbformat

SKIP_MARKER = "90_publish_snapshot.ipynb skips this cell"

nb = nbformat.reads(requests.get(NOTEBOOK_URL, timeout=60).text, as_version=4)

sources, skipped = [], []
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    if src.lstrip().startswith("%%"):           # %%bigquery cells need loaded tables
        skipped.append("magic")
        continue
    if SKIP_MARKER in src:                      # Appendix B and anything else opted out
        skipped.append("marked")
        continue
    if "load_table(" in src:                    # keep the cell, drop the BigQuery writes
        src = re.sub(r"^\s*load_table\(.*?\)\s*$", "", src, flags=re.M | re.S)
    sources.append(src)

# --- Find the config cell, once, and prove it is unambiguous ----------------
# The student cell reads:
#     DEFAULT_METRO = "Chicago"
#     METRO = DEFAULT_METRO
# We rewrite the second line, not the first, so DEFAULT_METRO keeps its meaning
# and the "you are on the default" nudge only fires on the run that really is.
METRO_LINE = re.compile(r'(?m)^METRO\s*=\s*.+$')     # not METROS - no '=' after 'METRO'

config_idx = [i for i, s in enumerate(sources)
              if "DEFAULT_METRO" in s and METRO_LINE.search(s)]
if len(config_idx) != 1:
    raise RuntimeError(
        f"Expected exactly one config cell assigning METRO; found {len(config_idx)}: "
        f"{config_idx}. The student notebook's config cell has changed shape and the "
        f"metro override below would not work. Fix this before publishing anything."
    )
CONFIG_IDX = config_idx[0]


def set_metro(src, metro):
    """Rewrite the config cell's METRO assignment. Raises rather than no-ops."""
    new, n = METRO_LINE.subn(f'METRO = "{metro}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"METRO substitution matched {n} lines, expected 1")
    return new


# Prove the substitution works before we spend an hour finding out it does not.
probe = set_metro(sources[CONFIG_IDX], "Denver")
assert 'METRO = "Denver"' in probe and "DEFAULT_METRO" in probe

print(f"Code cells in the student notebook : {sum(1 for c in nb.cells if c.cell_type == 'code')}")
print(f"Skipped (magic / marked)           : {skipped.count('magic')} / {skipped.count('marked')}")
print(f"Cells we will execute              : {len(sources)}")
print(f"Config cell                        : index {CONFIG_IDX}, substitution verified")

## Build and export, one metro at a time

Each metro runs in its own namespace so a failure in one cannot contaminate the next. We keep
going on failure and report at the end—one bad metro should not cost you the other eight.

Two things here are defensive rather than obvious, and both are the residue of a bad snapshot:

**We capture each dataframe the moment it exists**, rather than reaching into the namespace at
the end. A later cell that binds the same name to something else—an `int`, say—used to take a
table out of the snapshot silently. Capturing on sight makes the export independent of whatever
happens downstream of the cell that produced the data.

**We assert the metro actually changed** before uploading a single byte. `METRO` is read back
out of the executed namespace and compared to the metro we asked for. If a future edit breaks
the substitution again, this run stops on the first metro instead of publishing nine copies of
the same city.

In [ ]:
import io
import time
import traceback
import pandas as pd
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

results = {}

for metro, (state, _fips) in METROS.items():
    print(f"\n{'=' * 64}\n{metro}\n{'=' * 64}")
    t0 = time.time()
    ns = {"__name__": "__main__"}
    captured = {}
    try:
        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = set_metro(src, metro)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

            # Grab each table as soon as it appears. Whatever a later cell does
            # to the name afterwards is then none of our business.
            for var in TABLES.values():
                val = ns.get(var)
                if isinstance(val, pd.DataFrame):
                    captured[var] = val

        # --- Did the override actually take? ---------------------------------
        ran_as = ns.get("METRO")
        if ran_as != metro:
            raise RuntimeError(
                f"metro override failed: asked for {metro!r}, notebook ran as {ran_as!r}. "
                f"Nothing uploaded. Fix set_metro() before rerunning."
            )
        if ns.get("STATE") != state:
            raise RuntimeError(f"{metro} ran with STATE={ns.get('STATE')!r}, expected {state!r}")

        missing = [f"{t} ({v})" for t, v in TABLES.items() if v not in captured]
        if missing:
            raise RuntimeError("never saw a dataframe for: " + ", ".join(missing))

        slug = metro.lower().replace(" ", "-")
        for table, var in TABLES.items():
            df = captured[var]
            df = df.drop(columns=[c for c in ("tract_geom",) if c in df.columns])
            if df.empty:
                raise RuntimeError(f"{table} is empty - refusing to publish it")
            buf = io.BytesIO()
            df.to_parquet(buf, index=False)
            buf.seek(0)
            blob = bucket.blob(f"{PREFIX}/{slug}/{table}/data.parquet")
            blob.upload_from_file(buf, content_type="application/octet-stream")
            print(f"  {table:<22} {len(df):>7,} rows -> gs://{BUCKET}/{blob.name}")

        results[metro] = ("OK", time.time() - t0)
    except Exception as exc:                              # noqa: BLE001
        print(f"  FAILED: {exc}")
        traceback.print_exc()
        results[metro] = (f"FAILED: {exc}", time.time() - t0)

print(f"\n{'=' * 64}\nSUMMARY\n{'=' * 64}")
for metro, (status, secs) in results.items():
    print(f"  {metro:<16} {secs:>6.1f}s  {status}")

## Verify the snapshot is loadable—and is actually the city it claims to be

Publishing is not the same as publishing something that works, and something that works is not
the same as something that is right. This cell reads every file back the way `load.sh` will,
then asks four questions of it:

1. **Is it there, and does it have rows?** The floor.
2. **Profile variety.** The one number that decides whether vector search can work at all. A
   corpus where every profile is identical would load cleanly and rank meaninglessly.
3. **Is this the right city?** Organizations must be in the expected state, the metro's own
   name must appear among its cities, and every census tract id must begin with that state's
   FIPS code. This is the check that was missing when nine metros published Chicago's data—the
   row counts were all plausible, because a wrong city still has a believable number of
   pantries in it.
4. **Are the metros actually different from each other?** We fingerprint each metro's recipient
   set. Two metros with an identical fingerprint means the override collapsed again.

`shelf_life` is deliberately excluded from 3 and 4: it is USDA FoodKeeper, identical everywhere.

In [ ]:
import hashlib

problems = []
fingerprints = {}


def flag(metro, msg):
    problems.append(f"{metro}: {msg}")
    return "   <-- " + msg


for metro, (state, fips) in METROS.items():
    slug = metro.lower().replace(" ", "-")
    print(f"\n{metro}")
    for table in TABLES:
        path = f"{PREFIX}/{slug}/{table}/data.parquet"
        blob = bucket.blob(path)
        if not blob.exists():
            print(f"  {table:<20} MISSING  gs://{BUCKET}/{path}")
            problems.append(f"{metro}: {table} missing")
            continue

        df = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
        note = ""

        if len(df) == 0:
            note += flag(metro, f"{table} is empty")

        if table == "recipients":
            variety = df["profile_text"].nunique() / max(len(df), 1)
            note += f"  variety {variety:.0%}"
            if variety < 0.9:
                note += flag(metro, f"profiles only {variety:.0%} distinct - regenerate")

            states = df["state"].astype(str).str.strip().str.upper()
            top_state = states.mode().iat[0] if len(states) else "?"
            note += f"  state {top_state}"
            if top_state != state:
                note += flag(metro, f"organizations are in {top_state}, expected {state}")

            cities = df["city"].astype(str).str.strip().str.upper()
            if metro.upper() not in set(cities):
                note += flag(metro, f"no organization has city == {metro.upper()!r} "
                                    f"(top city is {cities.mode().iat[0]!r})")

            fingerprints[metro] = hashlib.sha1(
                "".join(sorted(df["ein"].astype(str))).encode()).hexdigest()[:12]

        if table == "tract_demographics":
            gid = df["geo_id"].astype(str)
            bad_len = int((gid.str.len() != 11).sum())
            in_state = float((gid.str[:2] == fips).mean()) if len(gid) else 0.0
            top_fips = gid.str[:2].mode().iat[0] if len(gid) else "??"
            note += f"  FIPS {top_fips} ({in_state:.0%})"
            if bad_len:
                note += flag(metro, f"{bad_len} tract ids are not 11 characters")
            # A metro bounding box is allowed to cross a state line - Camden sits
            # in the Philadelphia box, Newark in the New York one, and those tracts
            # belong there. What is not allowed is the majority being elsewhere.
            if top_fips != fips or in_state < 0.5:
                note += flag(metro, f"only {in_state:.0%} of tracts are in state FIPS "
                                    f"{fips} (majority is {top_fips}) - THIS IS THE WRONG CITY")

        print(f"  {table:<20} {len(df):>7,} rows{note}")

# --- Are any two metros the same data? -------------------------------------
print(f"\n{'=' * 64}\nRecipient fingerprints\n{'=' * 64}")
seen = {}
for metro, fp in fingerprints.items():
    dup = seen.get(fp)
    print(f"  {metro:<16} {fp}" + (f"   <-- IDENTICAL TO {dup}" if dup else ""))
    if dup:
        problems.append(f"{metro} and {dup} published identical recipient sets")
    seen[fp] = metro

print(f"\n{'=' * 64}")
if problems:
    print("Problems above. Do not announce this snapshot.\n")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"All good. {len(METROS)} metros, {len(TABLES)} tables each, "
          f"every one in the state it claims and distinct from the others.")